# 猫狗识别 · 标准答案

这份完整程序与题目版使用相同数据和流程，可以运行并对照各处代码。训练结果取决于设置与运行条件。

In [ ]:
from pathlib import Path
import os, json, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('project_outputs')
WORK.mkdir(exist_ok=True)
SIZE, BATCH, EPOCHS, LR = 128, 32, 5, 0.001  # 修改尺寸或轮数后，重新运行后续单元格
print('device:', DEVICE, 'output:', WORK)
def show(images, titles, path=None):
    fig, axes = plt.subplots(1,len(images),figsize=(4*len(images),4),squeeze=False)
    for ax, im, title in zip(axes[0],images,titles):
        if isinstance(im,torch.Tensor): im=im.detach().cpu().permute(1,2,0).numpy()
        ax.imshow(im); ax.set_title(title); ax.axis('off')
    fig.tight_layout()
    if path: fig.savefig(path,dpi=150,bbox_inches='tight')
    plt.show(); plt.close(fig)
def ready(*items):
    ok = all(item is not Ellipsis for item in items)
    if not ok: print('本部分仍有 ... 待填；补全后重新运行本格及后续代码。')
    return ok

RUN_TRAINING = True  # 补全代码后改为 True；训练 A、B 两个方案

## 数据与划分

Input 中已添加完整 Oxford-IIIT Pet 数据。代码读取官方 `trainval.txt` 和 `test.txt`，共 7349 张有官方划分的照片；其余未进入这两份清单的图片不加入实验。官方测试集保留不变，trainval 按猫狗比例划出 20% 验证集。

来源：Parkhi et al., Cats and Dogs, CVPR 2012；Oxford-IIIT Pet，CC BY-SA 4.0。

In [ ]:
INPUT = Path(os.environ.get('KYDW_PET_ROOT','/kaggle/input'))
roots=[INPUT] if (INPUT/'annotations/trainval.txt').exists() else [p.parent.parent for p in INPUT.rglob('trainval.txt') if p.parent.name=='annotations']
assert roots, '请在 Add Input 中添加 the-oxfordiiit-pet-dataset，包含 images 和 annotations。'
ROOT=roots[0]
def read_split(name):
    rows=[]
    for line in (ROOT/'annotations'/f'{name}.txt').read_text().splitlines():
        if not line.strip() or line.startswith('#'): continue
        image,breed,species,*_=line.split()
        rows.append(dict(id=image,label=int(species)-1,breed=int(breed),image=str(ROOT/'images'/f'{image}.jpg'),mask=str(ROOT/'annotations/trimaps'/f'{image}.png')))
    return pd.DataFrame(rows)
trainval, test = read_split('trainval'), read_split('test')
train, val = train_test_split(trainval,test_size=0.2,random_state=SEED,stratify=trainval.label)
train,val,test=[d.reset_index(drop=True) for d in [train,val,test]]
assert not (set(train.id)&set(val.id) or set(trainval.id)&set(test.id))
all_rows=pd.concat([train.assign(split='train'),val.assign(split='val'),test.assign(split='test')])
assert all(Path(p).is_file() for p in all_rows.image), '存在缺失照片'
assert all(Path(p).is_file() for p in all_rows['mask']), '存在缺失分割标注'
all_rows[['id','label','breed','split']].to_csv(WORK/'split.csv',index=False)
print(all_rows.groupby(['split','label']).size().unstack(fill_value=0).rename(columns={0:'cat',1:'dog'}))
print('官方 trainval:',len(trainval),'官方 test:',len(test),'实际使用总数:',len(all_rows))

## 照片、标签与尺寸分布

同一张照片对应整图猫狗标签和逐像素 trimap。修改 `SAMPLE_INDEX` 可以查看另一只动物；下面的直方图统计训练照片的原始尺寸。

In [ ]:
SAMPLE_INDEX = 0  # 改成另一行编号，观察照片、原始尺寸与标签
row=train.iloc[SAMPLE_INDEX]
photo=Image.open(row.image).convert('RGB'); mask=Image.open(row['mask'])
print('id:',row.id,'size:',photo.size,'species:',['cat','dog'][row.label])
show([photo,np.asarray(mask)],['Original photo','Trimap: 1 pet / 2 background / 3 boundary'])
sizes=np.array([Image.open(p).size for p in train.image])
fig,axes=plt.subplots(1,2,figsize=(10,3))
axes[0].hist(sizes[:,0],bins=30,label='width',alpha=.6);axes[0].hist(sizes[:,1],bins=30,label='height',alpha=.6);axes[0].legend();axes[0].set_xlabel('Original pixels')
train.label.value_counts().sort_index().plot.bar(ax=axes[1]);axes[1].set_xticklabels(['cat','dog'],rotation=0);axes[1].set_ylabel('Training images');plt.tight_layout();plt.show();plt.close(fig)

## 填空 1 · 图像数值

照片以 0—255 的整数保存。`SCALE` 应让处理后的颜色位于 0—1；在下面填写除数，然后核对输出范围与显示颜色。

In [ ]:
SCALE = 255.0  # 待填：数值缩放的除数
class PetDataset(Dataset):
    def __init__(self,rows,flip=0): self.rows=rows;self.flip=flip
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]
        im=Image.open(row.image).convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        if random.random()<self.flip: im=im.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        x=torch.from_numpy(np.asarray(im).copy()).permute(2,0,1).float()/SCALE
        return x,int(row.label),row.id
if ready(SCALE):
    assert isinstance(SCALE,(int,float)) and SCALE>0
    x,y,_=PetDataset(train)[0];print('CHW:',x.shape,'range:',x.min().item(),x.max().item(),'label:',y)
    show([x],['Resized model input'])

## 填空 2 · 分类网络

三个卷积块提取图像特征，最后的线性层给每个类别输出一个原始分数。猫和狗分别占一列；填写类别数量。损失使用交叉熵，网络末尾不添加 Softmax。

网络中的 `BatchNorm2d` 在训练时按批次调整中间特征的数值尺度，预测时使用训练中保存的统计量；`train()` 与 `eval()` 分别控制两种状态。

In [ ]:
NUM_CLASSES = 2  # 待填：输出类别数量
def make_model():
    return nn.Sequential(nn.Conv2d(3,16,3,padding=1),nn.BatchNorm2d(16),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.BatchNorm2d(32),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(32,64,3,padding=1),nn.BatchNorm2d(64),nn.ReLU(),nn.MaxPool2d(2),nn.AdaptiveAvgPool2d(4),nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Linear(64,NUM_CLASSES)).to(DEVICE)
if ready(NUM_CLASSES):
    assert isinstance(NUM_CLASSES,int) and NUM_CLASSES>0
    print('Output shape:',make_model()(torch.zeros(2,3,SIZE,SIZE,device=DEVICE)).shape)

## 填空 3 · 损失与预测

交叉熵接收 `[B,类别数]` 的原始分数和 `[B]` 的整数标签。填写 PyTorch 的交叉熵函数（函数本身，不加括号），以及预测时类别所在的轴编号。

In [ ]:
LOSS_FN = F.cross_entropy  # 待填：交叉熵函数
CLASS_AXIS = 1  # 待填：[批量,类别] 中类别轴

## 训练与验证

训练循环已经包含梯度清零、前向计算、损失、反向传播和参数更新。每轮在验证集计算损失并保存最低损失权重，测试集不参与权重选择。A 不翻转，B 以 0.5 的概率水平翻转，其余条件相同。

In [ ]:
def epoch(model,loader,optimizer=None):
    model.train(optimizer is not None);total,count=0.,0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in loader:
            x,target=batch[0].to(DEVICE),batch[1].to(DEVICE)
            pred=model(x)
            loss=LOSS_FN(pred,target)
            if optimizer is not None:
                optimizer.zero_grad();loss.backward();optimizer.step()
            total+=loss.item()*len(x);count+=len(x)
    return total/count

def run_experiment(name,flip):
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    model=make_model()
    optimizer=torch.optim.Adam(model.parameters(),lr=LR)
    train_loader=DataLoader(PetDataset(train,flip),batch_size=BATCH,shuffle=True,num_workers=0)
    val_loader=DataLoader(PetDataset(val),batch_size=BATCH,num_workers=0)
    history=[];best=float('inf');folder=WORK/name;folder.mkdir(exist_ok=True)
    for step in range(EPOCHS):
        a=epoch(model,train_loader,optimizer);b=epoch(model,val_loader)
        history.append(dict(epoch=step+1,train_loss=a,val_loss=b));print(name,history[-1])
        if b<best:
            best=b;torch.save(model.state_dict(),folder/'best.pt')
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    (folder/'config.json').write_text(json.dumps(dict(seed=SEED,size=SIZE,batch=BATCH,epochs=EPOCHS,lr=LR,flip=flip,train=len(train),val=len(val),test=len(test)),indent=2))
    model.load_state_dict(torch.load(folder/'best.pt',map_location=DEVICE,weights_only=True));model.eval()
    return model,folder
models={}
if RUN_TRAINING and ready(SCALE,NUM_CLASSES,LOSS_FN,CLASS_AXIS):
    for name,flip in [('A',0.0),('B',0.5)]: models[name]=run_experiment(name,flip)
else: print('补全待填处后，将 RUN_TRAINING 改为 True，再运行本格。A 与 B 只改变水平翻转概率。')

## 测试与错误样例

混淆矩阵区分两类误判。错误照片与预测表保留相同样本编号；对照 A、B 的结果，说明增强改变了哪些判断。测试结果用于最后一次评价，后续调参仍使用验证集。

In [ ]:
for name,(model,folder) in models.items():
    rows=[]
    with torch.no_grad():
        for x,y,ids in DataLoader(PetDataset(test),batch_size=BATCH):
            scores=model(x.to(DEVICE));pred=scores.argmax(dim=CLASS_AXIS).cpu()
            rows.extend(dict(id=i,truth=int(t),prediction=int(p)) for i,t,p in zip(ids,y,pred))
    table=pd.DataFrame(rows);table.to_csv(folder/'predictions.csv',index=False)
    print(name,'test n:',len(table),'accuracy:',(table.truth==table.prediction).mean())
    ConfusionMatrixDisplay(confusion_matrix(table.truth,table.prediction,labels=[0,1]),display_labels=['cat','dog']).plot();plt.savefig(folder/'confusion.png',dpi=150);plt.show();plt.close()
    hist=pd.read_csv(folder/'history.csv');hist.plot(x='epoch',y=['train_loss','val_loss']);plt.savefig(folder/'curves.png',dpi=150);plt.show();plt.close()
    errors=table[table.truth!=table.prediction].head(4)
    if len(errors):
        pics=[Image.open(test.set_index('id').loc[i,'image']).convert('RGB') for i in errors.id]
        show(pics,[f'{q.id}: {q.truth} -> {q.prediction}' for q in errors.itertuples()],folder/'errors.png')
if len(models)==2:
    a=pd.read_csv(WORK/'A/predictions.csv');b=pd.read_csv(WORK/'B/predictions.csv');comparison=a.merge(b,on='id',suffixes=('_A','_B'),validate='one_to_one');comparison.to_csv(WORK/'comparison.csv',index=False)

## PPT · 深度学习流程图与结果页

制作两页可编辑幻灯片：一页画出图像处理、网络、类别分数，以及标签、损失和参数更新；一页放实际预测与一个指标。画布 16:9，中文微软雅黑、英文 Arial，标题 28—32 磅、框内文字 18—22 磅、图注 14—16 磅。同层级字体字号一致，用对齐与分布统一间距，使用吸附到形状的连接符，照片保持比例。保存 `流程图与结果.pptx` 和 PDF。

## 保存项目

下载本次修改后的 Notebook 和 Output 中的结果。`split.csv` 记录实际划分，A、B 文件夹保存权重、设置、曲线及逐图结果。将代码、主要结果和 PPT 或 Word 放进对应项目文件夹。